# SQL Explorer

Run any SQL query against the v2 database. Just edit the query string and run the cell.

In [ ]:
import sys, os
os.chdir(os.path.expanduser("~/alpha-signal-v2"))
sys.path.insert(0, ".")

import pandas as pd
from db import read_sql, get_db
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

def sql(query, limit=50):
    """Run a SQL query and display results. Add LIMIT if not present."""
    q = query.strip().rstrip(";")
    if limit and "limit" not in q.lower():
        q += f" LIMIT {limit}"
    try:
        return read_sql(q)
    except Exception as e:
        print(f"SQL Error: {e}")
        print(f"Query: {q}")
        print("\nHint: SQLite uses DISTINCT (not UNIQUE), || for concat, substr() not substring()")
        return None

def tables():
    """Show all tables with columns and row counts."""
    with get_db() as conn:
        tbls = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name != 'sqlite_sequence' ORDER BY name"
        ).fetchall()
    
    rows = []
    for (tbl,) in tbls:
        with get_db() as conn:
            count = conn.execute(f"SELECT COUNT(*) FROM [{tbl}]").fetchone()[0]
            cols_info = conn.execute(f"PRAGMA table_info([{tbl}])").fetchall()
        col_names = [c[1] for c in cols_info]
        col_types = [c[2] for c in cols_info]
        rows.append({
            "table": tbl,
            "rows": count,
            "columns": len(col_names),
            "column_names": ", ".join(col_names),
            "column_types": ", ".join(col_types),
        })
    return pd.DataFrame(rows)

def describe(table):
    """Show column names, types, and sample values for a table."""
    with get_db() as conn:
        cols_info = conn.execute(f"PRAGMA table_info([{table}])").fetchall()
    
    sample = read_sql(f"SELECT * FROM [{table}] LIMIT 3")
    rows = []
    for c in cols_info:
        col_name = c[1]
        rows.append({
            "column": col_name,
            "type": c[2],
            "nullable": "YES" if not c[3] else "NO",
            "pk": "PK" if c[5] else "",
            "sample": str(sample[col_name].iloc[0])[:60] if col_name in sample.columns and len(sample) > 0 else "—",
        })
    return pd.DataFrame(rows)

print("Ready. Commands:")
print("  sql('SELECT ...')     — run any query")
print("  tables()              — all tables with columns")
print("  describe('stocks')    — columns + types + sample for one table")

## All tables with columns

In [ ]:
tables()

## Data Health Dashboard

In [ ]:
from db import data_health
data_health()

## Query anything

Edit the SQL below and run:

In [ ]:
sql("SELECT cap_tier, COUNT(*) as n FROM stocks GROUP BY cap_tier")

In [ ]:
sql("SELECT * FROM regime_state")

In [ ]:
sql("SELECT * FROM daily_picks ORDER BY cap_tier, rank LIMIT 15")